# Spatially Masked Regression on AJILE12: mask sweep and electrode coverage

Ostadsharif Memar & Dehghani (2026), arXiv:2606.11415, on DANDI:000055 version 0.220127.0436
(the only published version; see `reproduce_memar_2026.ipynb` for the header, the version choice
and the full data check). Claims tested here: fig4a-mask-sweep-ieeg and fig5a-coverage-ieeg.
Settings and deviations as in `reproduce_memar_2026.ipynb` (every 10th sample for the gradient
steps, 200 epochs; events from `ReachEvents`). Each claim is run under both readings of where the
preprocessing is applied, (a) before segmentation and (b) per epoch.

In [ ]:
import os
for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"          # one linear-algebra thread per process; the fits run in parallel
import time, zlib, json, resource
import numpy as np
import pandas as pd
from scipy import stats
import smr

T0 = time.time()
PAPER = "memar-2026"
def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())

DECIM, MAX_EPOCH = 10, 200
SUBJECTS = [f"sub-{i:02d}" for i in range(1, 13)]
if os.environ.get("SMR_TEST"):                 # development only: two subjects
    SUBJECTS = ["sub-07", "sub-04"]
IDX = [int(s[-2:]) - 1 for s in SUBJECTS]
print("dataset", smr.DANDISET, smr.VERSION)

## Data check

The trials of every subject, preprocessed as in `reproduce_memar_2026.ipynb` under both readings (its data check covers the metadata, duplicates and the clock).

In [ ]:
t = time.time()
DATA = {}
for s in SUBJECTS:
    S = smr.load_subject(s)
    DATA[s] = smr.drop_uncoordinated(S)
    x = DATA[s]["a"]
    print(s, "sessions", len(S["sessions"]), "trials", x.shape[0], "electrodes", x.shape[1], "samples", x.shape[2],
          "dropped without coordinates", DATA[s]["n_dropped_el"], f"range {x.min():.2f} to {x.max():.2f}, SD {x.std():.3f}",
          f"reading b range {DATA[s]['b'].min():.2f} to {DATA[s]['b'].max():.2f} ({time.time() - t:.0f} s)", flush=True)
print("trials in all", sum(S["a"].shape[0] for S in DATA.values()))
print("peak memory MB", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3)

In [ ]:
def run_all(m, rng, reading="a", workers=3, **kw):
    # one SMR condition on every subject, in parallel; each subject's generator is seeded from rng
    seeds = rng.integers(2**63, size=len(SUBJECTS))
    jobs = [(smr.run_smr, (DATA[s][reading], smr.neighbours(DATA[s]["xyz"]), m, np.random.default_rng(k)),
             dict(decim=DECIM, max_epoch=MAX_EPOCH, **kw)) for s, k in zip(SUBJECTS, seeds)]
    return dict(zip(SUBJECTS, smr.parallel(jobs, workers)))

def subj(res):
    return np.array([res[s]["dc"].mean() for s in SUBJECTS])

def msd(v):
    return f"{np.mean(v):.3f} ± {np.std(v):.3f}"

## fig4a-mask-sweep-ieeg

> In iEEG, DistCorr decreases as more of the target's local neighborhood is masked (mask
> intensity 0, 25, 50, 75 and 100%).

ceil(m × 9) neighbours drawn at random per electrode (Results III D), m in 0, 0.25, 0.5, 0.75 and 1.0, one generator for the whole sweep of each preprocessing reading (the claim's for reading a, its third for reading b). For time, reading (b) is run at m = 0 and m = 1.0 only (the two ends, which the coverage claim also uses). The code's rule, round(m × 9), differs at m = 0.25 (2 in place of 3) and m = 0.5 (4 in place of 5), and is shown as a variant.

In [ ]:
t = time.time()
SW = {}
MS = {"a": (0.0, 0.25, 0.5, 0.75, 1.0), "b": (0.0, 1.0)}
for prep, add in (("a", 0), ("b", 2)):
    rng_sweep = np.random.default_rng(seed("fig4a-mask-sweep-ieeg") + add)
    SWEEP = {}
    for m in MS[prep]:
        SWEEP[m] = run_all(m, rng_sweep, prep)
        print(f"reading {prep}, m = {m} done ({time.time() - t:.0f} s)", flush=True)
    SW[prep] = SWEEP
sw_by = {}
for prep in ("a", "b"):
    ms = list(MS[prep])
    sw = np.array([subj(SW[prep][m]) for m in ms])          # (m, subject)
    sw_by[prep] = sw
    print(f"--- preprocessing reading {prep}")
    print(pd.DataFrame(sw.T.round(3), index=SUBJECTS, columns=[f"m={m}" for m in ms]).to_string())
    avg = sw.mean(1)
    print("average over subjects:", dict(zip(ms, avg.round(3))))
    steps = np.diff(avg)
    print("average decreases at every step:", bool((steps < 0).all()), "steps", steps.round(4).tolist())
    mono = (np.diff(sw, axis=0) < 0).all(0)
    print("subjects decreasing at every step:", int(mono.sum()), "of", len(SUBJECTS), [s for s, k in zip(SUBJECTS, mono) if not k])
    print("m = 1.0 below m = 0:", avg[-1] < avg[0], f"({avg[-1]:.3f} against {avg[0]:.3f})")
    print("m = 1.0 values (full-mask-above-zero-ieeg):", dict(zip(SUBJECTS, sw[-1].round(3))))
    print("median epoch of the best validation loss per m:", {m: int(np.median(np.concatenate([SW[prep][m][s]["best_ep"] for s in SUBJECTS]))) for m in ms})
ms = list(MS["a"])
print("masked neighbours per electrode:", {m: int(np.ceil(m * 9 - 1e-9)) for m in ms})
sw = sw_by["a"]

Variant: the code's round(m × 9) at m = 0.25 and 0.5 (round 1 of this attempt: averages 0.685 and 0.660, falling at every step in all 12 subjects) is not run in this revision, for time.

In [ ]:
print("round variant not run in this revision")

**Result: reproduced.** Under reading (a) the subject average falls at every step, 0.714, 0.673, 0.651, 0.628 and 0.602 from m = 0 to m = 1.0, and in each of the 12 subjects. Under reading (b), run at the two ends only, it falls from 0.659 to 0.571, in 12 of 12 subjects.

## fig5a-coverage-ieeg

> In iEEG, reconstruction is best with all electrodes as predictors, and local electrodes alone
> consistently outperform non-local electrodes alone.

All = only the target masked (m = 0 of the sweep); Non-local = target and neighbourhood masked
(m = 1.0 of the sweep); Local = only the 9 neighbours as predictors (Results III E; run_coverage_experiment.py).
Under both preprocessing readings (the claim's generator for reading a, its second for reading b).

In [ ]:
t = time.time()
LOCAL = {}
for prep, add in (("a", 0), ("b", 1)):
    LOCAL[prep] = run_all(0.0, np.random.default_rng(seed("fig5a-coverage-ieeg") + add), prep, mode="local")
    print(f"local, reading {prep} done ({time.time() - t:.0f} s); median epoch of the best validation loss "
          f"{int(np.median(np.concatenate([LOCAL[prep][s]['best_ep'] for s in SUBJECTS])))}", flush=True)
for prep in ("a", "b"):
    cov = pd.DataFrame({"subject": SUBJECTS, "all": subj(SW[prep][0.0]), "local": subj(LOCAL[prep]),
                        "non_local": subj(SW[prep][1.0])})
    print(f"--- preprocessing reading {prep}")
    print(cov.round(3).to_string(index=False))
    a_, l_, n_ = cov["all"].mean(), cov["local"].mean(), cov["non_local"].mean()
    print(f"averages: all {a_:.3f}, local {l_:.3f}, non-local {n_:.3f}; ordered all > local > non-local: {a_ > l_ > n_}")
    print("subjects with local above non-local:", int((cov.local > cov.non_local).sum()), "of", len(SUBJECTS), ";",
          "with all above local:", int((cov["all"] > cov.local).sum()), "of", len(SUBJECTS))

**Result: not reproduced.** All is highest under both readings (0.714 and 0.659, 12 of 12 subjects), but Local alone is below Non-local on average: 0.567 against 0.602 under reading (a), Local above in 2 of 12 subjects, and 0.569 against 0.571 under reading (b), Local above in 4 of 12. The order All > Local > Non-local does not hold under either reading. Movement-event reading (b) was not run, so the outcome rests on reading (a) of the events.

## Summary

| Claim | Outcome |
|---|---|
| fig4a-mask-sweep-ieeg | reproduced (0.714 to 0.602 under preprocessing a, falling at every step in 12 of 12 subjects; 0.659 to 0.571 under b) |
| fig5a-coverage-ieeg | not reproduced (Local below Non-local: 0.567 against 0.602 under a, 0.569 against 0.571 under b) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method. The differences and the dataset findings
are listed in the summary of `reproduce_memar_2026.ipynb`. For time, reading (b) of the sweep
was run at m = 0 and m = 1.0 only, and the round variant was not run in this revision.


In [ ]:
print(f"notebook ran {(time.time() - T0) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")